In [0]:
# Step 1: Read CSV files from the Volume
file_path = "/Volumes/dev_finance/staging/data_export/csv/"

# Read CSV files with header and infer schema
# multiLine=true handles multi-line text fields properly
# escape="\"" handles escaped quotes within text
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .load(file_path)

print(f"Total records: {df.count()}")
print(f"Schema:")
df.printSchema()
display(df.limit(5))

In [0]:
# Step 10: Verify the loaded data and show table properties
# Query the most recent table in bronze schema

from pyspark.sql.functions import col

# List all review tables in bronze schema
tables_df = spark.sql("""
    SHOW TABLES IN dev_finance.bronze 
    LIKE 'reviews_*'
""")

print("Tables in dev_finance.bronze:")
display(tables_df)

# Get the most recent table (if you want to query it)
if tables_df.count() > 0:
    latest_table = tables_df.orderBy(col("tableName").desc()).first()["tableName"]
    full_table_name = f"dev_finance.bronze.{latest_table}"
    
    print(f"\nQuerying latest table: {full_table_name}")
    
    # Show table details
    spark.sql(f"DESCRIBE EXTENDED {full_table_name}").show(50, truncate=False)
    
    # Show sample data with aggregations
    print("\nSummary statistics:")
    spark.sql(f"""
        SELECT 
            COUNT(*) as total_records,
            COUNT(DISTINCT record_id) as unique_reviews,
            COUNT(DISTINCT franchise_id) as unique_franchises,
            AVG(review_length) as avg_review_length,
            MIN(review_year) as earliest_year,
            MAX(review_year) as latest_year,
            COUNT(DISTINCT review_category) as review_categories
        FROM {full_table_name}
    """).show()

In [0]:
# Step 2: Using select() to choose specific columns
# select() allows you to pick columns by name, create new columns, or apply functions

# Example: Select specific columns from the DataFrame
df_selected = df.select("review", "franchiseID", "review_date", "new_id")

print("Selected columns using select():")
display(df_selected.limit(10))

In [0]:
# Step 3: Using col() and lit() functions
# col() - references a DataFrame column
# lit() - creates a literal/constant value column

from pyspark.sql.functions import col, lit

# Using col() to reference columns and lit() to add constant values
df_with_col_lit = df.select(
    col("review"),
    col("franchiseID"),
    col("review_date"),
    col("new_id"),
    lit("Processed").alias("status"),  # Add constant value
    lit("2026-08-21").alias("load_date")  # Add load date
)

print("Using col() and lit() functions:")
display(df_with_col_lit.limit(10))

In [0]:
# Step 4: Using selectExpr() to write SQL-like expressions
# selectExpr() allows you to use SQL syntax for transformations
# try_cast handles malformed data gracefully by returning NULL

df_select_expr = df.selectExpr(
    "review",
    "franchiseID",
    "review_date",
    "new_id",
    "upper(review) as review_upper",  # Convert review to uppercase
    "length(review) as review_length",  # Calculate review text length
    "concat('FRAN-', franchiseID) as franchise_code",  # Create franchise code
    "try_cast(review_date as timestamp) as review_timestamp",  # Convert string to timestamp
    "year(try_cast(review_date as timestamp)) as review_year",  # Extract year from date
    "current_timestamp() as processed_timestamp"  # Add timestamp
)

print("Using selectExpr() with SQL expressions:")
display(df_select_expr.limit(10))

In [0]:
# Step 5: Using withColumn() to add or modify columns
# withColumn() adds a new column or replaces an existing one

from pyspark.sql.functions import col, lit, current_timestamp, upper, concat, when, length, year, expr

df_with_columns = df \
    .withColumn("review_timestamp", expr("try_cast(review_date as timestamp)")) \
    .withColumn("review_length", length(col("review"))) \
    .withColumn("review_category", 
                when(col("review_length") < 50, "Short")
                .when(col("review_length") < 200, "Medium")
                .otherwise("Long")) \
    .withColumn("franchise_code", concat(lit("FRAN-"), col("franchiseID"))) \
    .withColumn("review_year", year(col("review_timestamp"))) \
    .withColumn("load_timestamp", current_timestamp()) \
    .withColumn("source_system", lit("CSV_IMPORT"))

print("Using withColumn() to add calculated and constant columns:")
display(df_with_columns.limit(10))

In [0]:
# Step 6: Using withColumnRenamed() to rename columns
# withColumnRenamed() renames existing columns

df_renamed = df_with_columns \
    .withColumnRenamed("review", "review_text") \
    .withColumnRenamed("franchiseID", "franchise_id") \
    .withColumnRenamed("review_date", "submitted_date") \
    .withColumnRenamed("new_id", "record_id") \
    .withColumnRenamed("review_length", "text_length")

print("After renaming columns:")
df_renamed.printSchema()
display(df_renamed.limit(10))

In [0]:
# Step 7: Add file metadata columns
# _metadata.file_path captures the source file path for Unity Catalog

from pyspark.sql.functions import current_timestamp, regexp_extract, col

df_with_metadata = df_with_columns \
    .withColumn("source_file_path", col("_metadata.file_path")) \
    .withColumn("source_file_name", 
                regexp_extract(col("_metadata.file_path"), r"([^/]+)$", 1)) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("data_source", lit("volume_csv"))

print("DataFrame with file metadata:")
display(df_with_metadata.select(
    "review", "franchiseID", "review_date", "new_id",
    "source_file_name", "source_file_path", "ingestion_timestamp"
).limit(10))

In [0]:
# Step 8: Create final transformed DataFrame with all transformations
# Combine all transformations in a single pipeline

from pyspark.sql.functions import col, lit, current_timestamp, upper, concat, when, regexp_extract, md5, length, year, month, dayofmonth, expr

df_final = df.select(
    col("new_id").alias("record_id"),
    col("review").alias("review_text"),
    col("franchiseID").alias("franchise_id"),
    col("review_date")
) \
.withColumn("review_timestamp", expr("try_cast(review_date as timestamp)")) \
.withColumn("review_length", length(col("review_text"))) \
.withColumn("review_category", 
            when(col("review_length") < 50, "Short")
            .when(col("review_length") < 200, "Medium")
            .otherwise("Long")) \
.withColumn("franchise_code", concat(lit("FRAN-"), col("franchise_id"))) \
.withColumn("review_year", year(col("review_timestamp"))) \
.withColumn("review_month", month(col("review_timestamp"))) \
.withColumn("source_file", regexp_extract(col("_metadata.file_path"), r"([^/]+)$", 1)) \
.withColumn("ingestion_timestamp", current_timestamp()) \
.withColumn("processing_id", md5(concat(col("record_id").cast("string"), col("ingestion_timestamp").cast("string")))) \
.withColumn("status", lit("Processed")) \
.withColumn("source_system", lit("CSV_VOLUME"))

print(f"Final DataFrame - Total records: {df_final.count()}")
df_final.printSchema()
display(df_final.limit(10))

In [0]:
# Step 9: Load data into dev_finance.bronze catalog
# CREATE OR REPLACE TABLE to create new table each time

import time

# Generate unique table name with timestamp
table_name = f"dev_finance.bronze.reviews_{int(time.time())}"

print(f"Creating table: {table_name}")

# Write DataFrame to Delta table
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✓ Successfully created table: {table_name}")
print(f"✓ Total records loaded: {spark.table(table_name).count()}")

# Display sample data from the new table
print("\nSample data from bronze table:")
display(spark.table(table_name).limit(10))